# Distributed Biometric Risk Categorization — Analysis Notebook

**Objective:** classify biometric records into risk categories with a Random Forest classifier (Python, Pandas, NumPy, Scikit-learn), and evaluate the model with accuracy, precision, recall, F1-score, a confusion matrix and ROC-AUC.

This notebook reuses the modules in `src/`, so it runs exactly the same code as the command-line pipeline.

> **Dataset note:** by default this notebook points at the **synthetic demo dataset** in `data/sample/`, which exists only to show the pipeline working. To analyse your real data, change `DATA_PATH` and `TARGET_COLUMN` in the configuration cell below. Any numbers shown with the demo data are **not** project results.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import DEFAULT_RF_PARAMS, DOCUMENTED_RESULTS
from src.data_preprocessing import clean_data, inspect_data, load_data, split_features_target, split_train_test
from src.evaluate import compare_with_documented, evaluate_model
from src.feature_engineering import identify_column_types
from src.train import resolve_positive_label, train_model

## 1. Configuration

In [ ]:
DATA_PATH = ROOT / "data" / "sample" / "sample_data.csv"   # <- change to your CSV, e.g. ROOT / "data" / "raw" / "your_data.csv"
TARGET_COLUMN = "risk_category"                           # <- change to your target column
POSITIVE_LABEL = None                                     # binary targets: None = last class in sorted order
RANDOM_STATE = 42

## 2. Dataset loading and overview

In [ ]:
df = load_data(DATA_PATH)
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

In [ ]:
summary = inspect_data(df, TARGET_COLUMN)
display(pd.DataFrame({"dtype": summary["dtypes"], "missing": summary["missing_values"]}))
print("Duplicate rows:", summary["duplicate_rows"])
print("Class distribution:", summary["class_distribution"])

## 3. Preprocessing

Rows with a missing target are dropped. Missing feature values are imputed **inside** the scikit-learn pipeline (median for numeric columns, most-frequent for non-numeric columns), so the same imputation is applied at prediction time. The split is stratified and uses a fixed random seed.

In [ ]:
df_clean = clean_data(df, TARGET_COLUMN)
X, y = split_features_target(df_clean, TARGET_COLUMN)
X_train, X_test, y_train, y_test = split_train_test(X, y, random_state=RANDOM_STATE)
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

## 4. Feature preparation

In [ ]:
numeric_cols, categorical_cols = identify_column_types(X_train)
print("Numeric columns    :", numeric_cols)
print("Non-numeric columns:", categorical_cols)

## 5. Model training (Random Forest, no hyperparameter tuning)

In [ ]:
print("Random Forest settings:", DEFAULT_RF_PARAMS, "| random_state =", RANDOM_STATE)
pipeline = train_model(X_train, y_train, random_state=RANDOM_STATE)
classes = pipeline.classes_.tolist()
bundle = {"pipeline": pipeline, "positive_label": resolve_positive_label(classes, POSITIVE_LABEL)}
print("Classes:", classes)

## 6. Predictions

In [ ]:
predictions = pd.DataFrame({"actual": y_test.values, "predicted": pipeline.predict(X_test)})
for cls, probs in zip(classes, pipeline.predict_proba(X_test).T):
    predictions[f"probability_{cls}"] = probs.round(3)
display(predictions.head(10))

## 7. Evaluation

In [ ]:
metrics = evaluate_model(bundle, X_test, y_test)
print("Precision/Recall/F1 averaging:", metrics["average"])
display(pd.Series({k: metrics[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]}, name="value").to_frame())

## 8. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(pipeline, X_test, y_test, cmap="Blues")
plt.title("Confusion matrix (held-out test set)")
plt.show()

## 9. ROC-AUC analysis

In [ ]:
if len(classes) == 2:
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, pos_label=bundle["positive_label"])
    plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
    plt.legend()
    plt.title(f"ROC curve (AUC = {metrics['roc_auc']:.4f})")
    plt.show()
else:
    print(f"Multiclass target: one-vs-rest macro ROC-AUC = {metrics['roc_auc']}")

## 10. Comparison with the documented project result

In [ ]:
print("Documented project result (author-supplied):", DOCUMENTED_RESULTS)
comparison = compare_with_documented(metrics)
if comparison is None:
    print("Not comparable for this target/dataset.")
else:
    print("Reproduced by this run? ", comparison)

## 11. Conclusion

This notebook walked through loading the data, cleaning it, preparing features, training a Random Forest, and evaluating it on a held-out test set using accuracy, precision, recall, F1-score, a confusion matrix and ROC-AUC.

Whether the documented project result (Accuracy 99.31%, Recall 99.79%, ROC-AUC 0.9997) is reproduced depends on running this notebook against the original 200K+ record dataset; section 10 reports that honestly. With the bundled synthetic demo data, the numbers only show that the pipeline works and say nothing about the original project.